In [1]:
import sys
import os

target_dir = os.path.abspath('../src/')
sys.path.insert(0, target_dir) 

In [3]:
from sortedcontainers import SortedDict
from orderbook import OrderBook
ob = OrderBook()

ob.bids= SortedDict({-99.8: 40, -99.7: 40, -99.6: 100})
ob.asks= SortedDict({100.0: 20, 100.1: 30})

### Signal: The Micro-Price (The "True" Mid)
It tells "Expected next price"

**Logic:** The simple mid-price ($S = \frac{Bid + Ask}{2}$) ignores the "pressure" in the book. If there are 100 orders on the Bid and only 1 on the Ask, the price is likely to move **up**. The Micro-price accounts for this.

**The Math:**
We weight the Bid and Ask prices by their **opposite** volumes:


$$MicroPrice = \frac{Price_{Bid} \cdot Vol_{Ask} + Price_{Ask} \cdot Vol_{Bid}}{Vol_{Bid} + Vol_{Ask}}$$

**Intuition:** * If $Vol_{Bid}$ is huge, the $MicroPrice$ gets pulled toward $Price_{Ask}$.

* It is a "high-frequency" leading indicator. If $MicroPrice > MidPrice$, the next trade is statistically more likely to be a "Buy."

In [4]:
bid, bid_vol = ob.bids.peekitem(0)
ask, ask_vol = ob.asks.peekitem(0)
bid = abs(bid)
print('bid, bid_vol, ask, ask_vol = ', bid, bid_vol, ask, ask_vol)

bid, bid_vol, ask, ask_vol =  99.8 40 100.0 20


In [5]:
micro_price = (bid*ask_vol+ask*bid_vol)/(ask_vol+bid_vol)
micro_price

99.93333333333334

In [6]:
# reduce ask_vol = 5
ask_vol = 5
micro_price = (bid*ask_vol+ask*bid_vol)/(ask_vol+bid_vol)
micro_price # closer to 100.0, looks ask side being fragile!

99.97777777777777

### 2. Signal : Order Book Imbalance (OBI)
Tells Which side is "heavier."
**Logic:** This quantifies the "buy-sell pressure" into a single oscillator. It is the primary signal for adjusting your **Reservation Price** (how you skew your quotes).

**The Math:**


$$I = \frac{Vol_{Bid} - Vol_{Ask}}{Vol_{Bid} + Vol_{Ask}}$$

* **Range:** $[-1, 1]$
* **$I = +1$:** All volume is on the Bid (Strong Bullish).
* **$I = -1$:** All volume is on the Ask (Strong Bearish).
* **$I = 0$:** Perfect equilibrium.

**Application:** When $I$ is high, you move your quotes up. You want to make it harder for people to buy from you (since the price is going up) and easier for people to sell to you (to get you into a long position before the rally).

In [7]:
I = (bid_vol-ask_vol)/(bid_vol+ask_vol)
I

0.7777777777777778

In [8]:
ask_vol = 1000
I = (bid_vol-ask_vol)/(bid_vol+ask_vol)
I

-0.9230769230769231

### 3. Kyle’s Lambda ($\lambda$) - The "Toxicity" Signal
Tells how "toxic" the flow is. | Price change $\div$ Trade volume. |
**Logic:** It measures **Adverse Selection**. If you are a Market Maker and a large trader "sweeps" your book, they probably know something you don't. $\lambda$ tells you how much the price moves per unit of volume traded.

**The Math:**
In a simple linear model:


$$\Delta P = \lambda \cdot (V_{buy} - V_{sell}) + \epsilon$$

To implement this rolling in your project, we calculate the average price impact over the last $N$ trades:


$$\lambda_{rolling} = \frac{1}{N} \sum_{i=1}^{N} \frac{|Price_{i} - Price_{i-1}|}{Volume_i}$$

**The "Defense" Logic:**

* **Low $\lambda$:** The market is liquid and "noise-driven." You can keep your spreads tight and make money on the volume.
* **High $\lambda$:** The market is "toxic." Every trade is moving the price significantly.
* **Action:** You must **widen your spread**. If you don't, you'll be "picked off" (buying right before the price drops or selling right before it moons).


In [9]:
# which price to consider? micro or mid -> micro is better!

In [10]:
del_price = 0.5
del_vol = 5000-10
kl = (del_price/del_vol)
kl

0.0001002004008016032

### 4. Signal: Realized Volatility ($\sigma$)
Tells how "risky" the asset is. 
**Logic:** This is your "Risk Clock." Volatility tells you the probability that the price will move outside your spread before you can hedge your position.

**The Math:**
Since we are dealing with high-frequency tick data, we use the **Standard Deviation of Log Returns** of the Micro-price over a sliding window ($W$):


$$r_t = \ln\left(\frac{P_t}{P_{t-1}}\right)$$

$$\sigma = \sqrt{\frac{\sum(r_t - \bar{r})^2}{W-1}}$$

**Why it matters for Avellaneda-Stoikov:**
The "Optimal Spread" formula is directly proportional to $\sigma^2$. If volatility doubles, your spread must widen by more than double to compensate for the inventory risk.


In [21]:
import numpy as np

# 20 Micro-price updates
prices = [
    100.00, 100.02, 100.01, 100.03, 100.00, # Stable
    100.05, 100.10, 100.12, 100.08, 100.15, # Trending
    100.40, 100.80, 101.50, 101.30, 101.90, # Breakout/Toxic
    101.85, 101.90, 101.88, 101.92, 101.90  # High-level stability
]

def calculate_vol(price_window):
    if len(price_window) < 2: return 0
    # Log Returns: ln(Pt / Pt-1)
    returns = np.diff(np.log(price_window))
    return np.std(returns)


vol_quiet = calculate_vol(prices[0:5])
vol_breakout = calculate_vol(prices[10:15])
vol_overall = calculate_vol(prices)

print("quiet=", vol_quiet, " vol_breakout=", vol_breakout, " overall=", vol_overall)

quiet= 0.0002121002207950181  vol_breakout= 0.0034455632319346217  overall= 0.0022019043561539355
